# 00 - Setup & Mount

Mount Google Drive, install `uv`, and verify the Drive project root is reachable.

Run order: this notebook first, then `01_build_data_layers.ipynb`.

In [ ]:
import os, sys
from pathlib import Path

# ── environment detection ────────────────────────────────────────────────────
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ── Google Drive mount (Colab only) ──────────────────────────────────────────
# force_remount=False: skips re-auth if already mounted in this runtime session.
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    _DEFAULT_ROOT = '/content/drive/MyDrive/陽交大高速公路計畫/main'
else:
    # Walk up from CWD to find project root (works whether kernel starts at
    # workspace root or inside notebooks/).
    _here = Path(os.getcwd()).resolve()
    for _p in [_here, *_here.parents]:
        if (_p / 'CLAUDE.md').exists():
            _DEFAULT_ROOT = str(_p)
            break
    else:
        _DEFAULT_ROOT = str(_here)

PROJECT_ROOT = os.environ.get('PROJECT_ROOT', _DEFAULT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)

In [ ]:
%cd {PROJECT_ROOT}
!ls -la

In [ ]:
# Install uv. The data scripts use inline /// script /// dependency blocks.
!pip install -q uv
!uv --version

In [ ]:
# Sanity-check the data tree.
import json, sys
from pathlib import Path
sys.path.insert(0, str(Path(PROJECT_ROOT)))

for p in [
    'data/vd_raw_lane_1m_full',
    'data/vd_station_5m',
    'data/historical_speed_profile_5m',
    'data/accident_samples_5m.npz',
    'data/accident_data.csv',
    'data/metadata/station_build_report.json',
]:
    full = Path(PROJECT_ROOT) / p
    print(f'{full.exists():>5}  {p}')

In [ ]:
# Smoke-import the shared library.
from scripts.lib import SampleWindow, DEFAULT_WINDOW
from scripts.lib.data_loaders import duckdb_connection, read_accidents
print('DEFAULT_WINDOW =', DEFAULT_WINDOW.to_dict())
print('Matrix shape   =', (DEFAULT_WINDOW.n_space, DEFAULT_WINDOW.n_time))

In [ ]:
# Optional: open a DuckDB connection over the project parquet datasets.
con = duckdb_connection()
print('Registered views:')
for row in con.sql("SELECT table_name FROM information_schema.tables ORDER BY table_name").fetchall():
    print(' -', row[0])